# Dubai Real Estate — Feature Engineering

Clean, reproducible feature-engineering layer for the Dubai Real Estate AVM.

The final XGBoost valuation model uses these 12 features:

- `log_area_sqft`
- `bedrooms`
- `parking_count`
- `is_offplan`
- `is_freehold`
- `area_90d_median_ppsf`
- `project_90d_median_ppsf`
- `log_area_90d_count`
- `log_project_90d_count`
- `project_history_available`
- `AREA_EN`
- `PROJECT_EN`

Design principles: point-in-time comparable history, no forward leakage, semantic treatment of missing history, and an explicit final feature contract.

## 1. Imports and configuration

Expected source file: `data/raw/transactions-2026-08-17.csv`.

Keep raw transaction data out of GitHub unless redistribution is permitted.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "transactions-2026-08-17.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SQFT_PER_SQM = 10.763910416709722

REQUIRED_RAW_COLUMNS = [
    "TRANSACTION_NUMBER", "INSTANCE_DATE", "AREA_EN", "PROJECT_EN",
    "ROOMS_EN", "IS_OFFPLAN_EN", "IS_FREE_HOLD_EN", "TRANS_VALUE",
    "ACTUAL_AREA", "PARKING"
]

MODEL_FEATURES = [
    "log_area_sqft", "bedrooms", "parking_count", "is_offplan",
    "is_freehold", "area_90d_median_ppsf",
    "project_90d_median_ppsf", "log_area_90d_count",
    "log_project_90d_count", "project_history_available",
    "AREA_EN", "PROJECT_EN"
]

print("Raw source:", RAW_FILE)
print("Processed output:", PROCESSED_DIR)

## 2. Load and validate raw transactions

The notebook validates the minimum schema before transforming anything. This prevents silent column mismatches.

In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw transaction file not found: {RAW_FILE}")

raw = pd.read_csv(RAW_FILE)
missing = [c for c in REQUIRED_RAW_COLUMNS if c not in raw.columns]

if missing:
    raise ValueError(f"Missing required raw columns: {missing}")

print("Raw shape:", raw.shape)
print("Required schema: PASSED")

## 3. Clean types and identifiers

Missing values are not automatically removed. Missing comparable history can be valid information: it may mean there simply were not enough recent transactions to calculate a statistic.

In [ ]:
df = raw.copy()

df["INSTANCE_DATE"] = pd.to_datetime(df["INSTANCE_DATE"], errors="coerce")

for col in ["TRANS_VALUE", "ACTUAL_AREA"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in [
    "TRANSACTION_NUMBER", "AREA_EN", "PROJECT_EN", "ROOMS_EN",
    "IS_OFFPLAN_EN", "IS_FREE_HOLD_EN", "PARKING"
]:
    df[col] = df[col].astype("string").str.strip()

df = df.dropna(subset=["INSTANCE_DATE"]).copy()
df["row_id"] = np.arange(len(df), dtype=np.int64)
df = df.sort_values(["INSTANCE_DATE", "row_id"]).reset_index(drop=True)

print("Working shape:", df.shape)
print("Duplicate transaction IDs:", int(df["TRANSACTION_NUMBER"].duplicated().sum()))

## 4. Core property features

`ACTUAL_AREA` is converted from square metres to square feet. Price per square foot is then calculated from transaction value.

Studios are encoded as 0 bedrooms. Parking is parsed from the original text field. Unexpected off-plan/freehold labels are not silently coerced.

In [ ]:
df["area_sqft"] = df["ACTUAL_AREA"] * SQFT_PER_SQM
df["price_per_sqft"] = df["TRANS_VALUE"] / df["area_sqft"]
df["log_area_sqft"] = np.where(
    df["area_sqft"] > 0,
    np.log(df["area_sqft"]),
    np.nan,
)

room_text = df["ROOMS_EN"].astype("string")
df["bedrooms"] = pd.to_numeric(
    room_text.str.extract(r"(\d+)")[0],
    errors="coerce",
)
df.loc[
    room_text.str.contains("studio", case=False, na=False),
    "bedrooms",
] = 0

df["parking_count"] = pd.to_numeric(
    df["PARKING"].astype("string").str.extract(
        r"(-?\d+(?:\.\d+)?)"
    )[0],
    errors="coerce",
)

offplan_map = {
    "off-plan": 1,
    "off plan": 1,
    "offplan": 1,
    "ready": 0,
}

offplan_text = (
    df["IS_OFFPLAN_EN"]
    .astype("string")
    .str.lower()
    .str.strip()
)

unexpected_offplan = sorted(
    set(offplan_text.dropna().unique()) - set(offplan_map)
)

if unexpected_offplan:
    raise ValueError(
        f"Unexpected IS_OFFPLAN_EN values: {unexpected_offplan}"
    )

df["is_offplan"] = offplan_text.map(offplan_map)

def parse_freehold(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if "non" in text and "free" in text:
        return 0
    if "free" in text:
        return 1
    if text in {"yes", "y", "true", "1"}:
        return 1
    if text in {"no", "n", "false", "0"}:
        return 0
    return np.nan

df["is_freehold"] = df["IS_FREE_HOLD_EN"].map(parse_freehold)

print(
    df[
        [
            "area_sqft", "price_per_sqft", "log_area_sqft",
            "bedrooms", "parking_count", "is_offplan", "is_freehold"
        ]
    ].head()
)

## 5. Build point-in-time 90-day comparable features

For each transaction at time **T**, comparable features use only transactions from the previous 90 days.

The rolling window uses `closed='left'`, excluding observations at the current timestamp. This is the main leakage-control step.

We calculate history at both area and project level.

In [ ]:
def add_rolling_history(frame, group_col, prefix):
    frame = frame.copy()

    def one_group(group):
        group = group.sort_values(["INSTANCE_DATE", "row_id"]).copy()
        series = group.set_index("INSTANCE_DATE")["price_per_sqft"]
        rolling = series.rolling("90D", closed="left")

        group[f"{prefix}_90d_median_ppsf"] = rolling.median().to_numpy()
        group[f"{prefix}_90d_transaction_count"] = rolling.count().to_numpy()
        return group

    return (
        frame
        .groupby(group_col, group_keys=False, dropna=False)
        .apply(one_group)
        .reset_index(drop=True)
    )

df = add_rolling_history(df, "AREA_EN", "area")
df = add_rolling_history(df, "PROJECT_EN", "project")

print(
    df[
        [
            "INSTANCE_DATE", "AREA_EN", "PROJECT_EN", "price_per_sqft",
            "area_90d_median_ppsf", "area_90d_transaction_count",
            "project_90d_median_ppsf", "project_90d_transaction_count"
        ]
    ].head(10)
)

## 6. Comparable counts, log counts, and project-history availability

A zero recent-comparable count means **no recent evidence**, so it is represented as missing rather than fabricated as a numerical market signal.

`project_history_available = 1` only when the transaction date is later than the first historical date for that project. This allows a project to have history while still having no recent 90-day activity.

In [ ]:
area_count = df["area_90d_transaction_count"].astype(float).mask(lambda s: s.eq(0))
project_count = df["project_90d_transaction_count"].astype(float).mask(lambda s: s.eq(0))

df["area_90d_transaction_count"] = area_count
df["project_90d_transaction_count"] = project_count
df["log_area_90d_count"] = np.log1p(area_count)
df["log_project_90d_count"] = np.log1p(project_count)

first_project_date = (
    df.groupby("PROJECT_EN", dropna=False)["INSTANCE_DATE"]
    .transform("min")
)

df["project_history_available"] = (
    df["INSTANCE_DATE"].gt(first_project_date).astype(int)
)

print(
    df[
        [
            "PROJECT_EN", "INSTANCE_DATE", "project_history_available",
            "project_90d_transaction_count", "log_project_90d_count"
        ]
    ].head(10)
)

## 7. Preserve area/project as categorical features

The training data used pandas categorical columns for `AREA_EN` and `PROJECT_EN`. This notebook reproduces that representation with `.astype('category')`.

In [ ]:
df["AREA_EN"] = df["AREA_EN"].astype("category")
df["PROJECT_EN"] = df["PROJECT_EN"].astype("category")

print("AREA categories:", len(df["AREA_EN"].cat.categories))
print("PROJECT categories:", len(df["PROJECT_EN"].cat.categories))

## 8. Final model feature matrix

This is the exact feature contract used by the valuation model. Training and prediction code should consume this output instead of rebuilding features independently.

In [ ]:
X = df[MODEL_FEATURES].copy()

assert list(X.columns) == MODEL_FEATURES
assert X["AREA_EN"].dtype.name == "category"
assert X["PROJECT_EN"].dtype.name == "category"

print("X shape:", X.shape)
print("\nFeature dtypes:")
print(X.dtypes)
print("\nFeature schema: PASSED")

## 9. Feature-quality summary

Missing comparable-history features are intentionally reported, not automatically deleted. This distinguishes missing evidence from corrupt transaction data.

In [ ]:
quality = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_count": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

print(quality)

## 10. Leakage checks

Comparable features were generated with `closed='left'`, so current/future transactions are excluded from the historical window.

This notebook does **not** use future 30-day information. Future comparables belong only to the separate backtest.

In [ ]:
assert not df["INSTANCE_DATE"].isna().any()
assert not (df["price_per_sqft"] < 0).fillna(False).any()

print("No-future-feature rule: enforced by closed='left' rolling windows.")
print("Core leakage checks: PASSED")

## 11. Save the engineered dataset

The raw data remains outside the repository when redistribution is not allowed. The engineered file can be generated locally from the raw export.

In [ ]:
processed_columns = [
    "TRANSACTION_NUMBER", "INSTANCE_DATE", "AREA_EN", "PROJECT_EN",
    "ROOMS_EN", "IS_OFFPLAN_EN", "IS_FREE_HOLD_EN", "TRANS_VALUE",
    "ACTUAL_AREA", "area_sqft", "price_per_sqft", "log_area_sqft",
    "bedrooms", "parking_count", "is_offplan", "is_freehold",
    "area_90d_median_ppsf", "area_90d_transaction_count",
    "project_90d_median_ppsf", "project_90d_transaction_count",
    "log_area_90d_count", "log_project_90d_count",
    "project_history_available"
]

processed = df[processed_columns].copy()
output_file = PROCESSED_DIR / "transactions_feature_engineered.csv"
processed.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Shape:", processed.shape)

## 12. Handoff to modelling

The next stage can split `processed` chronologically into train/validation/test sets and train the valuation model.

Keep model training, model evaluation, opportunity scoring, forward backtesting, and Streamlit deployment in separate notebooks/scripts so this notebook stays focused and readable.